## Homework 6
----


#### Dataset
In this homework, we continue using the fuel efficiency dataset.

The goal of this homework is to create a regression model for predicting the car fuel efficiency (column `fuel_efficiency_mpg`)

#### Preparing the dataset
Preparation:
- Fill missing values with zeros.
- Do the split with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
```

- Use `DictVectorizer(sparse=True)` to turn the dataframes into matrices.

In [147]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb
%matplotlib inline

In [148]:
df = pd.read_csv('../02-regression/car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [149]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [150]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [151]:
df.horsepower = df.horsepower.fillna(0)
df.acceleration = df.acceleration.fillna(0)
df.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [152]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)
df_test = df_test.reset_index(drop = True)

y_train = df_train.fuel_efficiency_mpg
y_val = df_val.fuel_efficiency_mpg
y_test = df_test.fuel_efficiency_mpg

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [153]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

In [154]:
dv = DictVectorizer(sparse = True)

In [155]:
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

----
### Question 1
Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.
- Train a model with `max_depth = 1`.

Which original feature is used for splitting the data? If you use `DictVectorizer`, map a one-hot feature such as `origin=USA` back to its original column name.
- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

In [156]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [157]:
print(export_text(dt, feature_names=dv.get_feature_names_out()))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



**Ans:** `model_year`. 

According to the output from `export_text`, the feature `model_year` is used for splitting the data.

----
### Question 2
Train a random forest regressor with these parameters:
- `n_estimators = 10`
- `random_state = 1`
- `n_jobs = -1` (optional -- to make training faster)

What's the RMSE of this model on the validation data?
- 0.1837
- 1.837
- 18.837
- 183.7

In [158]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [159]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse(y_val, y_pred)

np.float64(1.8393827905033797)

**Ans:** 1.837

I got the RMSE = 0.1839, which is 0.0002 higher than the closest option: 1.837.

----

### Question 3
Now let's experiment with the `n_estimator` parameter
- Try these values of this parameter: `[10, 50, 100, 150]`.
- Set `random_state` to `1`
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 deciaml places for calculating the answer.
- 10
- 50 
- 100 
- 150

In [160]:
nums = [10, 50, 100, 150]
for n in nums:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    RMSE = rmse(y_val, y_pred)
    print(f'n_estimator = {n} | RMSE = {RMSE}')
    print('-------')

n_estimator = 10 | RMSE = 1.8393827905033797
-------
n_estimator = 50 | RMSE = 1.7728631357214235
-------
n_estimator = 100 | RMSE = 1.7676431541745072
-------
n_estimator = 150 | RMSE = 1.7681094485981979
-------


**Ans:** `n_estimator = 100`

According to the output above, `n_estimator = 100` brings the lowest RMSE = 1.7676

----

### Question 4
Let's select the best `max_depth`:
- Try different values of `max_depth`: `[10, 15, 20, 25]`
- For each of these values,
    - try different values of `n_estimators` from `[10, 50, 100, 150]`
    - Calculate the mean RMSE
- Fix the random seed: `random_state = 1`

What is the best `max_depth`, using the mean RMSE?
- 10
- 15
- 20
- 25

In [161]:
depths = [10, 15, 20, 25]
avg_RMSEs = []
for m in depths:
    curr_rmse = []
    for n in nums:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=m,
            random_state=1
            )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        RMSE = rmse(y_val, y_pred)
        curr_rmse.append(RMSE)
    avg_RMSEs.append(np.array(curr_rmse).mean())
    print(f'max_depth = {m} | avg_RMSE = {avg_RMSEs[-1]}')
    

max_depth = 10 | avg_RMSE = 1.754133023722073
max_depth = 15 | avg_RMSE = 1.7836702888451024
max_depth = 20 | avg_RMSE = 1.7878464776552963
max_depth = 25 | avg_RMSE = 1.786674552166829


**Ans:** `max_depth = 10`

According to the output above, `max_depth = 10` offers the lowest mean RMSE with the value of 1.754

----

### Question 5
We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" -- the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Sci-kit-Learn, tree-based models contain this information in the [`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_) field.

For this homework question, we'll find the most import feature:

- Train the model with these parameters:
    - `n_estimators=10`
    - `max_depth=20`
    - `random_state=1`
    - `n_jobs=-1` (optional)
- Get the feature importance information from this model

What is the most important feature (among these 4)?
- `vehicle_weight`
- `horsepower`
- `acceleration`
- `engine_displacement`

In [162]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
for imp, name in zip(rf.feature_importances_, dv.get_feature_names_out()):
    if name in ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']:
        print(f'feature: {name} | importance: {imp:.4f}')
    

feature: acceleration | importance: 0.0553
feature: engine_displacement | importance: 0.0612
feature: horsepower | importance: 0.0785
feature: vehicle_weight | importance: 0.2031


**Ans:** `vehicle_weight`

According to the output above, `vehicle_weight` has the largest importance (0.2031) among the 4 options

----

### Question 6
Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:
- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:

```python
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` for `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

- 0.3
- 0.1
- Both giver equal value

In [163]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label = y_train, feature_names = features)
dval = xgb.DMatrix(X_val, label = y_val, feature_names = features)

In [164]:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

In [165]:
model = xgb.train(xgb_params, dtrain, num_boost_round = 100)

In [166]:
y_pred = model.predict(dval)
RMSE = rmse(y_val, y_pred)

print(f'eta = 0.3 | RMSE = {RMSE:.4f}')

eta = 0.3 | RMSE = 1.8265


In [167]:
xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
model = xgb.train(xgb_params, dtrain, num_boost_round = 100)

y_pred = model.predict(dval)
RMSE = rmse(y_val, y_pred)

print(f'eta = 0.1 | RMSE = {RMSE:.4f}')

eta = 0.1 | RMSE = 1.7248


**Ans:** `eta = 0.1`

According to the outputs above, `eta = 0.1` has the lower `RMSE = 1.7248` compared to the one of `eta = 0.3`: `RMSE = 1.8265`

----
End of this page